# Multi-Step Water Inflow Forecasting - Model Development

Four forecasting approaches compared:

- **Holt-Winters (Triple Exponential Smoothing)**: Classical exponential smoothing with trend and seasonality
- **XGBoost**: Gradient-boosted trees with engineered lag features
- **Random Forest**: Bagging-based ensemble for variance reduction
- **LSTM**: Deep learning sequence model (PyTorch) for temporal dependencies

All evaluated with walk-forward (expanding window) CV using RMSE, MAE, MAPE, and MASE.

---

## 1. Setup and Configuration

In [1]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for memory safety

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Preprocessing and metrics
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

# Statistical models
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Machine learning
from xgboost import XGBRegressor

import warnings
import os
import time

warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Style and output
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

FIGURES_DIR = '../figures/'
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


---

## 2. Data Loading and Preparation

In [2]:
# Load data
df = pd.read_csv('../data/multistep_regression.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

Dataset shape: (281, 3)
Columns: ['Year', 'Month', 'Value']

First 5 rows:


,Year,Month,Value
0,1999,1,40.575599
1,1999,2,65.182165
2,1999,3,63.089448
3,1999,4,81.853452
4,1999,5,48.081737


In [3]:
# Create datetime index
df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(Day=1))
df = df.set_index('Date')
df = df.sort_index()
df.index.freq = 'MS'  # Month start frequency

# Rename for clarity
ts = df['Value'].copy()

print(f"Time range: {ts.index[0].strftime('%Y-%m')} to {ts.index[-1].strftime('%Y-%m')}")
print(f"Number of observations: {len(ts)}")
print(f"Missing values: {ts.isna().sum()}")
print(f"\nBasic statistics:")
ts.describe()

Time range: 1999-01 to 2022-05
Number of observations: 281
Missing values: 0

Basic statistics:


count    281.000000
mean      33.195956
std       25.893953
min        3.474676
25%       15.062259
50%       23.753365
75%       41.247543
max      139.154873
Name: Value, dtype: float64

In [4]:
# Train/Test split: December-anchored (assignment requirement)
# "Predictions must be generated every December for the subsequent five months"
TEST_SIZE = 5

# Final hold-out: train through Dec 2021, test Jan-May 2022
train_ts = ts[:'2021-12']
test_ts = ts['2022-01':'2022-05']

print(f"Training set: {train_ts.index[0].strftime('%Y-%m')} to {train_ts.index[-1].strftime('%Y-%m')} ({len(train_ts)} obs)")
print(f"Test set:     {test_ts.index[0].strftime('%Y-%m')} to {test_ts.index[-1].strftime('%Y-%m')} ({len(test_ts)} obs)")

# Visualize split
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_ts.index, train_ts.values, label='Train', color='steelblue', linewidth=1.5)
ax.plot(test_ts.index, test_ts.values, label='Test', color='coral', linewidth=1.5)
ax.axvline(x=test_ts.index[0], color='gray', linestyle='--', alpha=0.7, label='Split point (Dec 2021)')
ax.set_title('Water Inflow: Train/Test Split (December-Anchored, 5-Month Horizon)')
ax.set_xlabel('Date')
ax.set_ylabel('Water Inflow')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_train_test_split.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

Training set: 1999-01 to 2021-12 (276 obs)
Test set:     2022-01 to 2022-05 (5 obs)


18

### Feature Engineering Function

This function creates lag features, rolling statistics, and seasonal encodings for ML-based models.

In [5]:
def create_features(series, max_lag=12):
    """
    Create time series features from a univariate series.
    
    Parameters:
    -----------
    series : pd.Series with DatetimeIndex
    max_lag : int, maximum number of lag features
    
    Returns:
    --------
    pd.DataFrame with features
    """
    df_feat = pd.DataFrame(index=series.index)
    df_feat['Value'] = series.values
    
    # Lag features
    for lag in range(1, max_lag + 1):
        df_feat[f'lag_{lag}'] = series.shift(lag).values
    
    # Rolling statistics
    for window in [3, 6, 12]:
        df_feat[f'rolling_mean_{window}'] = series.shift(1).rolling(window=window).mean().values
    
    for window in [3, 6]:
        df_feat[f'rolling_std_{window}'] = series.shift(1).rolling(window=window).std().values
    
    # Seasonal encoding (cyclical)
    month = series.index.month
    df_feat['month_sin'] = np.sin(2 * np.pi * month / 12)
    df_feat['month_cos'] = np.cos(2 * np.pi * month / 12)
    
    # Time index (for trend)
    df_feat['time_index'] = np.arange(len(series))
    
    return df_feat


# Quick test
features_df = create_features(ts)
print(f"Features shape: {features_df.shape}")
print(f"Feature columns: {list(features_df.columns)}")
print(f"\nNaN counts (first 20 rows have NaNs due to lags):")
print(features_df.isna().sum()[features_df.isna().sum() > 0])

Features shape: (281, 21)
Feature columns: ['Value', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12', 'rolling_std_3', 'rolling_std_6', 'month_sin', 'month_cos', 'time_index']

NaN counts (first 20 rows have NaNs due to lags):
lag_1               1
lag_2               2
lag_3               3
lag_4               4
lag_5               5
lag_6               6
lag_7               7
lag_8               8
lag_9               9
lag_10             10
lag_11             11
lag_12             12
rolling_mean_3      3
rolling_mean_6      6
rolling_mean_12    12
rolling_std_3       3
rolling_std_6       6
dtype: int64


---

## 3. Walk-Forward Cross-Validation Framework

Expanding window CV -- training set grows each fold, anchored on December.
Predictions made every December for the subsequent 5 months (January-May).

| Fold | Train end  | Test period         | Train size |
|------|------------|---------------------|------------|
| 1    | Dec 2017   | Jan 2018 - May 2018 | 228        |
| 2    | Dec 2018   | Jan 2019 - May 2019 | 240        |
| 3    | Dec 2019   | Jan 2020 - May 2020 | 252        |
| 4    | Dec 2020   | Jan 2021 - May 2021 | 264        |
| 5    | Dec 2021   | Jan 2022 - May 2022 | 276        |

Fold 5 is the final hold-out test. Folds 1-4 provide cross-validated performance estimates.

In [6]:
def compute_mase(actual, predicted, train_series, seasonal_period=12):
    """
    Compute Mean Absolute Scaled Error.
    Scale factor is MAE of naive seasonal forecast on training data.
    """
    naive_errors = np.abs(train_series.values[seasonal_period:] - train_series.values[:-seasonal_period])
    scale = np.mean(naive_errors)
    if scale == 0:
        return np.nan
    mae = np.mean(np.abs(actual - predicted))
    return mae / scale


def compute_metrics(actual, predicted, train_series=None):
    """
    Compute RMSE, MAE, MAPE, and optionally MASE.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)
    
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = mean_absolute_percentage_error(actual, predicted) * 100  # as percentage
    
    metrics = {'RMSE': rmse, 'MAE': mae, 'MAPE': mape}
    
    if train_series is not None:
        mase = compute_mase(actual, predicted, train_series)
        metrics['MASE'] = mase
    
    return metrics


def get_cv_folds(series, first_test_year=2018, last_test_year=2022, test_months=5):
    """
    Generate December-anchored expanding window CV folds.
    
    Each fold trains through December of (test_year - 1) and tests on
    January through May of test_year.
    
    Returns
    -------
    list of (train_series, test_series, label) tuples
    """
    folds = []
    for test_year in range(first_test_year, last_test_year + 1):
        dec_date = pd.Timestamp(f'{test_year - 1}-12-01')
        may_date = pd.Timestamp(f'{test_year}-{test_months:02d}-01')
        jan_date = pd.Timestamp(f'{test_year}-01-01')
        
        if dec_date not in series.index or may_date not in series.index:
            continue
        
        train_fold = series[:dec_date]
        test_fold = series[jan_date:may_date]
        label = f'Dec {test_year - 1} -> Jan-May {test_year}'
        
        folds.append((train_fold, test_fold, label))
    
    return folds


# Generate folds
cv_folds = get_cv_folds(ts)

print("December-Anchored Walk-Forward CV Folds:")
print("=" * 80)
for i, (train_fold, test_fold, label) in enumerate(cv_folds):
    print(f"Fold {i+1}: {label} | "
          f"Train: {train_fold.index[0].strftime('%Y-%m')}-{train_fold.index[-1].strftime('%Y-%m')} ({len(train_fold)} obs) | "
          f"Test: {test_fold.index[0].strftime('%Y-%m')}-{test_fold.index[-1].strftime('%Y-%m')} ({len(test_fold)} obs)")

December-Anchored Walk-Forward CV Folds:
Fold 1: Dec 2017 -> Jan-May 2018 | Train: 1999-01-2017-12 (228 obs) | Test: 2018-01-2018-05 (5 obs)
Fold 2: Dec 2018 -> Jan-May 2019 | Train: 1999-01-2018-12 (240 obs) | Test: 2019-01-2019-05 (5 obs)
Fold 3: Dec 2019 -> Jan-May 2020 | Train: 1999-01-2019-12 (252 obs) | Test: 2020-01-2020-05 (5 obs)
Fold 4: Dec 2020 -> Jan-May 2021 | Train: 1999-01-2020-12 (264 obs) | Test: 2021-01-2021-05 (5 obs)
Fold 5: Dec 2021 -> Jan-May 2022 | Train: 1999-01-2021-12 (276 obs) | Test: 2022-01-2022-05 (5 obs)


---

## 4. Model 1: Holt-Winters (Triple Exponential Smoothing)

Holt-Winters decomposes the series into **level**, **trend**, and **seasonal** components, each updated with exponential smoothing. We use multiplicative seasonality with a period of 12 months. Parameters (alpha, beta, gamma) are optimized automatically via maximum likelihood.

In [7]:
# Holt-Winters Walk-Forward Cross-Validation
print("Holt-Winters Walk-Forward Cross-Validation")
print("=" * 70)

hw_fold_metrics = []
hw_fold_predictions = {}
hw_best = None

for fold_idx, (train_fold, test_fold, label) in enumerate(cv_folds):
    print(f"\n--- Fold {fold_idx + 1}: {label} ---")
    
    hw_configs = {}
    for trend in ['add', 'mul']:
        for seasonal in ['add', 'mul']:
            config_label = f'trend={trend}, seasonal={seasonal}'
            try:
                model = ExponentialSmoothing(
                    train_fold,
                    trend=trend,
                    seasonal=seasonal,
                    seasonal_periods=12
                ).fit(optimized=True)
                hw_configs[config_label] = {'model': model, 'aic': model.aic}
            except Exception:
                pass
    
    best_config = min(hw_configs, key=lambda k: hw_configs[k]['aic'])
    best_model = hw_configs[best_config]['model']
    
    fold_pred = best_model.forecast(steps=TEST_SIZE)
    fold_metrics = compute_metrics(test_fold.values, fold_pred.values, train_fold)
    
    hw_fold_metrics.append(fold_metrics)
    hw_fold_predictions[fold_idx] = {
        'pred': fold_pred, 'actual': test_fold,
        'train': train_fold, 'config': best_config
    }
    
    print(f"  Best config: {best_config}")
    for k, v in fold_metrics.items():
        print(f"  {k}: {v:.3f}")
    
    if fold_idx == len(cv_folds) - 1:
        hw_best = best_model

# CV Summary
hw_cv_summary = {}
for metric in ['RMSE', 'MAE', 'MAPE', 'MASE']:
    values = [m[metric] for m in hw_fold_metrics]
    hw_cv_summary[metric] = {'mean': np.mean(values), 'std': np.std(values)}

print("\n\nHolt-Winters CV Summary (across all folds):")
print("-" * 50)
for metric, stats in hw_cv_summary.items():
    print(f"  {metric}: {stats['mean']:.3f} +/- {stats['std']:.3f}")

# Final fold predictions for comparison plots
hw_pred = hw_fold_predictions[len(cv_folds) - 1]['pred']
hw_pred_series = pd.Series(hw_pred.values, index=test_ts.index)
hw_test_metrics = hw_fold_metrics[-1]

Holt-Winters Walk-Forward Cross-Validation

--- Fold 1: Dec 2017 -> Jan-May 2018 ---


  Best config: trend=add, seasonal=mul
  RMSE: 36.533
  MAE: 34.786
  MAPE: 119.051
  MASE: 2.347

--- Fold 2: Dec 2018 -> Jan-May 2019 ---


  Best config: trend=add, seasonal=mul
  RMSE: 15.314
  MAE: 12.343
  MAPE: 17.691
  MASE: 0.854

--- Fold 3: Dec 2019 -> Jan-May 2020 ---


  Best config: trend=add, seasonal=mul
  RMSE: 9.423
  MAE: 7.761
  MAPE: 11.568
  MASE: 0.521

--- Fold 4: Dec 2020 -> Jan-May 2021 ---


  Best config: trend=add, seasonal=mul
  RMSE: 16.190
  MAE: 13.774
  MAPE: 28.093
  MASE: 0.931

--- Fold 5: Dec 2021 -> Jan-May 2022 ---


  Best config: trend=add, seasonal=mul
  RMSE: 37.718
  MAE: 35.326
  MAPE: 62.219
  MASE: 2.403


Holt-Winters CV Summary (across all folds):
--------------------------------------------------
  RMSE: 23.036 +/- 11.743
  MAE: 20.798 +/- 11.811
  MAPE: 47.725 +/- 39.730
  MASE: 1.411 +/- 0.799


In [8]:
# Holt-Winters final fold results (Fold 5 = final hold-out test)
print("Holt-Winters - Final Test Set Performance (Fold 5: Jan-May 2022):")
for k, v in hw_test_metrics.items():
    print(f"  {k}: {v:.3f}")

print(f"\nSmoothing parameters: alpha={hw_best.params['smoothing_level']:.4f}, "
      f"beta={hw_best.params['smoothing_trend']:.4f}, "
      f"gamma={hw_best.params['smoothing_seasonal']:.4f}")

Holt-Winters - Final Test Set Performance (Fold 5: Jan-May 2022):
  RMSE: 37.718
  MAE: 35.326
  MAPE: 62.219
  MASE: 2.403

Smoothing parameters: alpha=0.7385, beta=0.0101, gamma=0.0000


In [9]:
# Holt-Winters Diagnostic Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted
ax = axes[0, 0]
ax.plot(train_ts.index[-36:], train_ts.values[-36:], label='Train (last 3 years)', 
        color='steelblue', linewidth=1.5)
ax.plot(test_ts.index, test_ts.values, label='Actual', color='coral', linewidth=2, marker='o', markersize=4)
ax.plot(hw_pred.index, hw_pred.values, label='Holt-Winters Forecast', 
        color='green', linewidth=2, linestyle='--', marker='s', markersize=4)
hw_resid_std = np.std(hw_best.resid)
ax.fill_between(test_ts.index, 
                hw_pred.values - 1.96 * hw_resid_std,
                hw_pred.values + 1.96 * hw_resid_std,
                alpha=0.2, color='green', label='95% CI')
ax.axvline(x=test_ts.index[0], color='gray', linestyle='--', alpha=0.5)
ax.set_title('Holt-Winters - Actual vs Predicted')
ax.legend(fontsize=9)
ax.set_ylabel('Water Inflow')

# 2. Residuals over time
residuals = hw_best.resid
ax = axes[0, 1]
ax.plot(residuals.index, residuals.values, color='steelblue', linewidth=0.8)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_title('Holt-Winters Residuals Over Time')
ax.set_ylabel('Residual')

# 3. Residual ACF
ax = axes[1, 0]
plot_acf(residuals.dropna(), ax=ax, lags=36, alpha=0.05)
ax.set_title('Holt-Winters Residual ACF')

# 4. Residual histogram
ax = axes[1, 1]
resid_clean = residuals.dropna()
ax.hist(resid_clean, bins=30, density=True, alpha=0.7, color='steelblue', edgecolor='black')
x = np.linspace(resid_clean.min(), resid_clean.max(), 100)
ax.plot(x, 1/(np.std(resid_clean)*np.sqrt(2*np.pi)) * np.exp(-0.5*((x-np.mean(resid_clean))/np.std(resid_clean))**2),
        'r--', linewidth=2, label='Normal fit')
ax.set_title('Holt-Winters Residual Distribution')
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_holtwinters_predictions.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

print(f'\nSmoothing parameters: alpha={hw_best.params["smoothing_level"]:.4f}, '
      f'beta={hw_best.params["smoothing_trend"]:.4f}, '
      f'gamma={hw_best.params["smoothing_seasonal"]:.4f}')


Smoothing parameters: alpha=0.7385, beta=0.0101, gamma=0.0000


---

## 5. Model 2: XGBoost for Time Series

Time series reformulated as tabular regression using lag features, rolling statistics, cyclical encoding, and trend index. Linear detrending applied first since trees cannot extrapolate beyond training range. Multi-step forecasting done recursively (each prediction fed back as lag for next step).

In [10]:
# Detrending helper
def fit_linear_trend(series):
    """Fit a linear trend to the series and return trend model, trend values, and residuals."""
    X = np.arange(len(series)).reshape(-1, 1)
    y = series.values
    lr = LinearRegression()
    lr.fit(X, y)
    trend = lr.predict(X)
    residual = y - trend
    return lr, trend, residual


def predict_trend(lr_model, start_idx, n_steps):
    """Predict linear trend for future steps."""
    X_future = np.arange(start_idx, start_idx + n_steps).reshape(-1, 1)
    return lr_model.predict(X_future)


# Feature creation for XGBoost with detrending
def create_xgb_features(series, detrended_values=None, max_lag=12):
    """
    Create features for XGBoost. Uses detrended values for lag/rolling features
    if provided, otherwise uses raw values.
    """
    vals = detrended_values if detrended_values is not None else series.values
    
    df_feat = pd.DataFrame(index=series.index)
    
    # Target (detrended)
    df_feat['target'] = vals
    
    # Lag features (on detrended values)
    for lag in range(1, max_lag + 1):
        df_feat[f'lag_{lag}'] = pd.Series(vals, index=series.index).shift(lag)
    
    # Rolling statistics (on detrended values, shifted by 1 to avoid leakage)
    detr_series = pd.Series(vals, index=series.index)
    for window in [3, 6, 12]:
        df_feat[f'rolling_mean_{window}'] = detr_series.shift(1).rolling(window=window).mean()
    
    for window in [3, 6]:
        df_feat[f'rolling_std_{window}'] = detr_series.shift(1).rolling(window=window).std()
    
    # Seasonal encoding (cyclical)
    month = series.index.month
    df_feat['month_sin'] = np.sin(2 * np.pi * month / 12)
    df_feat['month_cos'] = np.cos(2 * np.pi * month / 12)
    
    # Time index
    df_feat['time_index'] = np.arange(len(series))
    
    return df_feat


def recursive_predict(model, last_known_values, n_steps, trend_predictions,
                           last_train_features, future_dates):
    """
    Recursive multi-step prediction for sklearn-compatible regressors.
    Predict one step at a time, feeding predictions back as lag features.
    
    Parameters:
    -----------
    model : fitted sklearn-compatible regressor (XGBoost, Random Forest, etc.)
    last_known_values : array of last max_lag detrended values
    n_steps : number of steps to forecast
    trend_predictions : array of trend values for forecast period
    last_train_features : DataFrame row of the last training features (for rolling stats)
    future_dates : DatetimeIndex for forecast period
    """
    max_lag = 12
    predictions_detrended = []
    history = list(last_known_values[-max_lag:])  # Keep last max_lag detrended values
    
    feature_names = [f'lag_{i}' for i in range(1, max_lag + 1)] + \
                    ['rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
                     'rolling_std_3', 'rolling_std_6',
                     'month_sin', 'month_cos', 'time_index']
    
    for step in range(n_steps):
        features = {}
        
        # Lag features from history
        for lag in range(1, max_lag + 1):
            if lag <= len(history):
                features[f'lag_{lag}'] = history[-lag]
            else:
                features[f'lag_{lag}'] = 0  # Fallback
        
        # Rolling statistics from history
        hist_array = np.array(history)
        features['rolling_mean_3'] = np.mean(hist_array[-3:]) if len(hist_array) >= 3 else np.mean(hist_array)
        features['rolling_mean_6'] = np.mean(hist_array[-6:]) if len(hist_array) >= 6 else np.mean(hist_array)
        features['rolling_mean_12'] = np.mean(hist_array[-12:]) if len(hist_array) >= 12 else np.mean(hist_array)
        features['rolling_std_3'] = np.std(hist_array[-3:], ddof=1) if len(hist_array) >= 3 else 0
        features['rolling_std_6'] = np.std(hist_array[-6:], ddof=1) if len(hist_array) >= 6 else 0
        
        # Seasonal encoding
        month = future_dates[step].month
        features['month_sin'] = np.sin(2 * np.pi * month / 12)
        features['month_cos'] = np.cos(2 * np.pi * month / 12)
        
        # Time index
        features['time_index'] = len(last_known_values) + step
        
        # Create feature row
        X_step = pd.DataFrame([features])[feature_names]
        
        # Predict detrended value
        pred_detrended = model.predict(X_step)[0]
        predictions_detrended.append(pred_detrended)
        
        # Update history
        history.append(pred_detrended)
    
    # Add trend back
    predictions_final = np.array(predictions_detrended) + trend_predictions
    
    return predictions_final, np.array(predictions_detrended)


print("XGBoost helper functions defined.")

XGBoost helper functions defined.


In [11]:
# XGBoost Walk-Forward Cross-Validation
print("XGBoost Walk-Forward Cross-Validation")
print("=" * 70)

xgb_fold_metrics = []
xgb_fold_predictions = {}
xgb_final = None
X_train_final = None
y_train_final = None
residual_train_final = None
lr_final = None

for fold_idx, (train_fold, test_fold, label) in enumerate(cv_folds):
    print(f"\n--- Fold {fold_idx + 1}: {label} ---")
    
    # Detrend per fold
    lr_fold, trend_train, residual_train = fit_linear_trend(train_fold)
    
    # Create features
    feat_df = create_xgb_features(train_fold, detrended_values=residual_train)
    feat_df = feat_df.dropna()
    feature_cols = [c for c in feat_df.columns if c != 'target']
    X_train = feat_df[feature_cols]
    y_train = feat_df['target']
    
    # Fit
    xgb_model = XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=1, verbosity=0
    )
    xgb_model.fit(X_train, y_train)
    
    # Predict
    trend_test = predict_trend(lr_fold, len(train_fold), TEST_SIZE)
    fold_pred, _ = recursive_predict(
        xgb_model, residual_train, TEST_SIZE,
        trend_test, X_train.iloc[-1:], test_fold.index
    )
    
    fold_metrics = compute_metrics(test_fold.values, fold_pred, train_fold)
    xgb_fold_metrics.append(fold_metrics)
    xgb_fold_predictions[fold_idx] = {
        'pred': fold_pred, 'actual': test_fold, 'train': train_fold
    }
    
    for k, v in fold_metrics.items():
        print(f"  {k}: {v:.3f}")
    
    # Keep final fold artifacts
    if fold_idx == len(cv_folds) - 1:
        xgb_final = xgb_model
        X_train_final = X_train
        y_train_final = y_train
        residual_train_final = residual_train
        lr_final = lr_fold

# CV Summary
xgb_cv_summary = {}
for metric in ['RMSE', 'MAE', 'MAPE', 'MASE']:
    values = [m[metric] for m in xgb_fold_metrics]
    xgb_cv_summary[metric] = {'mean': np.mean(values), 'std': np.std(values)}

print("\n\nXGBoost CV Summary:")
print("-" * 50)
for metric, stats in xgb_cv_summary.items():
    print(f"  {metric}: {stats['mean']:.3f} +/- {stats['std']:.3f}")

# Final fold for comparison plots
xgb_pred = xgb_fold_predictions[len(cv_folds) - 1]['pred']
xgb_pred_series = pd.Series(xgb_pred, index=test_ts.index)
xgb_test_metrics = xgb_fold_metrics[-1]

XGBoost Walk-Forward Cross-Validation

--- Fold 1: Dec 2017 -> Jan-May 2018 ---
  RMSE: 3.689
  MAE: 2.884
  MAPE: 10.373
  MASE: 0.195

--- Fold 2: Dec 2018 -> Jan-May 2019 ---
  RMSE: 22.093
  MAE: 19.435
  MAPE: 24.981
  MASE: 1.345

--- Fold 3: Dec 2019 -> Jan-May 2020 ---


  RMSE: 29.704
  MAE: 24.305
  MAPE: 34.442
  MASE: 1.633

--- Fold 4: Dec 2020 -> Jan-May 2021 ---
  RMSE: 12.544
  MAE: 10.129
  MAPE: 30.056
  MASE: 0.684

--- Fold 5: Dec 2021 -> Jan-May 2022 ---
  RMSE: 21.449
  MAE: 19.763
  MAPE: 35.512
  MASE: 1.345


XGBoost CV Summary:
--------------------------------------------------
  RMSE: 17.896 +/- 8.946
  MAE: 15.303 +/- 7.733
  MAPE: 27.073 +/- 9.140
  MASE: 1.040 +/- 0.525


In [12]:
# XGBoost Diagnostic Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted
ax = axes[0, 0]
ax.plot(train_ts.index[-36:], train_ts.values[-36:], label='Train (last 3 years)', 
        color='steelblue', linewidth=1.5)
ax.plot(test_ts.index, test_ts.values, label='Actual', color='coral', linewidth=2, marker='o', markersize=4)
ax.plot(test_ts.index, xgb_pred, label='XGBoost Forecast', 
        color='green', linewidth=2, linestyle='--', marker='s', markersize=4)
ax.axvline(x=test_ts.index[0], color='gray', linestyle='--', alpha=0.5)
ax.set_title('XGBoost - Actual vs Predicted')
ax.legend(fontsize=9)
ax.set_ylabel('Water Inflow')

# 2. Feature Importance
ax = axes[0, 1]
importance = pd.Series(xgb_final.feature_importances_, index=feature_cols).sort_values(ascending=True)
importance.tail(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost Feature Importance (Top 15)')
ax.set_xlabel('Importance')

# 3. Residuals
xgb_residuals = test_ts.values - xgb_pred
ax = axes[1, 0]
ax.bar(range(len(xgb_residuals)), xgb_residuals, color='steelblue', alpha=0.7)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_title('XGBoost Test Set Residuals')
ax.set_xlabel('Forecast Horizon (months)')
ax.set_ylabel('Residual')
ax.set_xticks(range(len(xgb_residuals)))
ax.set_xticklabels([d.strftime('%Y-%m') for d in test_ts.index], rotation=45, fontsize=8)

# 4. Residual distribution
ax = axes[1, 1]
# In-sample residuals (training set)
xgb_train_pred = xgb_final.predict(X_train_final)
xgb_train_residuals = y_train_final.values - xgb_train_pred
ax.hist(xgb_train_residuals, bins=30, density=True, alpha=0.7, color='steelblue', edgecolor='black')
ax.set_title('XGBoost Training Residual Distribution')
ax.set_xlabel('Residual')

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_xgboost_predictions.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

15103

---

## 6. Model 3: Random Forest

Bagging-based ensemble — parallel independent trees reduce variance, complementing XGBoost's boosting (bias reduction).

In [13]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest Walk-Forward Cross-Validation
print("Random Forest Walk-Forward Cross-Validation")
print("=" * 70)

rf_fold_metrics = []
rf_fold_predictions = {}
rf_final = None

for fold_idx, (train_fold, test_fold, label) in enumerate(cv_folds):
    print(f"\n--- Fold {fold_idx + 1}: {label} ---")
    
    # Independent detrending per fold
    lr_fold, trend_train, residual_train = fit_linear_trend(train_fold)
    
    feat_df = create_xgb_features(train_fold, detrended_values=residual_train)
    feat_df = feat_df.dropna()
    feature_cols_rf = [c for c in feat_df.columns if c != 'target']
    X_train = feat_df[feature_cols_rf]
    y_train = feat_df['target']
    
    # Fit
    rf_model = RandomForestRegressor(
        n_estimators=100, max_depth=6, random_state=42, n_jobs=1
    )
    rf_model.fit(X_train, y_train)
    
    # Predict
    trend_test = predict_trend(lr_fold, len(train_fold), TEST_SIZE)
    fold_pred, _ = recursive_predict(
        rf_model, residual_train, TEST_SIZE,
        trend_test, X_train.iloc[-1:], test_fold.index
    )
    
    fold_metrics = compute_metrics(test_fold.values, fold_pred, train_fold)
    rf_fold_metrics.append(fold_metrics)
    rf_fold_predictions[fold_idx] = {
        'pred': fold_pred, 'actual': test_fold, 'train': train_fold
    }
    
    for k, v in fold_metrics.items():
        print(f"  {k}: {v:.3f}")
    
    if fold_idx == len(cv_folds) - 1:
        rf_final = rf_model

# CV Summary
rf_cv_summary = {}
for metric in ['RMSE', 'MAE', 'MAPE', 'MASE']:
    values = [m[metric] for m in rf_fold_metrics]
    rf_cv_summary[metric] = {'mean': np.mean(values), 'std': np.std(values)}

print("\n\nRandom Forest CV Summary:")
print("-" * 50)
for metric, stats in rf_cv_summary.items():
    print(f"  {metric}: {stats['mean']:.3f} +/- {stats['std']:.3f}")

# Final fold for comparison plots
rf_pred = rf_fold_predictions[len(cv_folds) - 1]['pred']
rf_pred_series = pd.Series(rf_pred, index=test_ts.index)
rf_test_metrics = rf_fold_metrics[-1]

Random Forest Walk-Forward Cross-Validation

--- Fold 1: Dec 2017 -> Jan-May 2018 ---
  RMSE: 3.069
  MAE: 2.905
  MAPE: 10.713
  MASE: 0.196

--- Fold 2: Dec 2018 -> Jan-May 2019 ---


  RMSE: 31.238
  MAE: 27.495
  MAPE: 34.368
  MASE: 1.903

--- Fold 3: Dec 2019 -> Jan-May 2020 ---
  RMSE: 16.313
  MAE: 13.636
  MAPE: 19.503
  MASE: 0.916

--- Fold 4: Dec 2020 -> Jan-May 2021 ---


  RMSE: 13.965
  MAE: 11.903
  MAPE: 32.677
  MASE: 0.804

--- Fold 5: Dec 2021 -> Jan-May 2022 ---
  RMSE: 25.725
  MAE: 21.203
  MAPE: 43.140
  MASE: 1.442


Random Forest CV Summary:
--------------------------------------------------
  RMSE: 18.062 +/- 9.767
  MAE: 15.428 +/- 8.384
  MAPE: 28.080 +/- 11.512
  MASE: 1.052 +/- 0.581


In [14]:
# Random Forest Diagnostic Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted
ax = axes[0, 0]
ax.plot(train_ts.index[-36:], train_ts.values[-36:], label='Train (last 3 years)',
        color='steelblue', linewidth=1.5)
ax.plot(test_ts.index, test_ts.values, label='Actual', color='coral', linewidth=2,
        marker='o', markersize=4)
ax.plot(test_ts.index, rf_pred, label='RF Forecast',
        color='#8B4513', linewidth=2, linestyle='--', marker='D', markersize=4)
ax.axvline(x=test_ts.index[0], color='gray', linestyle='--', alpha=0.5)
ax.set_title('Random Forest - Actual vs Predicted')
ax.legend(fontsize=9)
ax.set_ylabel('Water Inflow')

# 2. Feature Importance
ax = axes[0, 1]
rf_importance = pd.Series(rf_final.feature_importances_, index=feature_cols).sort_values(ascending=True)
rf_importance.tail(15).plot(kind='barh', ax=ax, color='#8B4513')
ax.set_title('Random Forest Feature Importance (Top 15)')
ax.set_xlabel('Importance')

# 3. Residuals
rf_residuals = test_ts.values - rf_pred
ax = axes[1, 0]
ax.bar(range(len(rf_residuals)), rf_residuals, color='#8B4513', alpha=0.7)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_title('Random Forest Test Set Residuals')
ax.set_xlabel('Forecast Horizon (months)')
ax.set_ylabel('Residual')
ax.set_xticks(range(len(rf_residuals)))
ax.set_xticklabels([d.strftime('%Y-%m') for d in test_ts.index], rotation=45, fontsize=8)

# 4. Residual distribution
ax = axes[1, 1]
rf_train_pred = rf_final.predict(X_train_final)
rf_train_residuals = y_train_final.values - rf_train_pred
ax.hist(rf_train_residuals, bins=30, density=True, alpha=0.7, color='#8B4513', edgecolor='black')
ax.set_title('Random Forest Training Residual Distribution')
ax.set_xlabel('Residual')

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_rf_walkforward.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

15282

In [15]:
# Feature Importance Comparison: XGBoost vs Random Forest
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# XGBoost importance
ax = axes[0]
xgb_imp = pd.Series(xgb_final.feature_importances_, index=feature_cols).sort_values(ascending=True)
xgb_imp.tail(10).plot(kind='barh', ax=ax, color='#DD8452', alpha=0.8)
ax.set_title('XGBoost Feature Importance (Top 10)', fontsize=12)
ax.set_xlabel('Importance (Gain)')

# Random Forest importance
ax = axes[1]
rf_imp = pd.Series(rf_final.feature_importances_, index=feature_cols).sort_values(ascending=True)
rf_imp.tail(10).plot(kind='barh', ax=ax, color='#8B4513', alpha=0.8)
ax.set_title('Random Forest Feature Importance (Top 10)', fontsize=12)
ax.set_xlabel('Importance (Impurity Decrease)')

plt.suptitle('Feature Importance: XGBoost vs Random Forest', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

# Correlation of feature importance rankings
from scipy.stats import spearmanr
common_features = set(xgb_imp.index) & set(rf_imp.index)
xgb_ranks = xgb_imp[list(common_features)].rank()
rf_ranks = rf_imp[list(common_features)].rank()
corr, pval = spearmanr(xgb_ranks, rf_ranks)
print(f'Spearman rank correlation of feature importances: {corr:.3f} (p={pval:.4f})')

Spearman rank correlation of feature importances: 0.638 (p=0.0025)


---

## 7. Model 4: LSTM (Long Short-Term Memory)

Deep learning sequence model — captures complex temporal dependencies through gated memory cells. Input is MinMax-scaled and shaped into lookback windows of 12 months. Recursive multi-step prediction (each output fed back as input).

In [16]:
import subprocess, pickle, os, sys, tempfile

# Run LSTM training in a separate process
# (PyTorch + Jupyter on macOS causes kernel crashes; subprocess avoids this)
print("LSTM Walk-Forward Cross-Validation")
print("=" * 70)
print("Running LSTM training in subprocess...")

output_path = os.path.join(tempfile.gettempdir(), 'lstm_results.pkl')
cmd = [
    sys.executable, '../src/lstm_trainer.py',
    '--data_path', '../data/multistep_regression.csv',
    '--output_path', output_path,
    '--test_size', str(TEST_SIZE),
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
# Logger writes to stderr (StreamHandler), so print both streams
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("LSTM training failed")

# Load results
with open(output_path, 'rb') as f:
    lstm_results = pickle.load(f)

lstm_fold_metrics = lstm_results['fold_metrics']
lstm_fold_predictions = lstm_results['fold_predictions']
lstm_cv_summary = lstm_results['cv_summary']
lstm_pred = lstm_results['final_pred']
lstm_test_metrics = lstm_results['final_metrics']

print("LSTM results loaded successfully.")

LSTM Walk-Forward Cross-Validation
Running LSTM training in subprocess...


2026-03-03 00:30:28 | INFO     | __main__ | LSTM Walk-Forward Cross-Validation (5 folds)
2026-03-03 00:30:28 | INFO     | __main__ | Fold 1/5: Dec 2017 -> Jan-May 2018
2026-03-03 00:30:29 | INFO     | __main__ |   Early stopped at epoch 34/150 (patience=10)
2026-03-03 00:30:29 | INFO     | __main__ |   Best val loss: 0.003302
2026-03-03 00:30:29 | INFO     | __main__ |   RMSE: 6.399
2026-03-03 00:30:29 | INFO     | __main__ |   MAE: 5.312
2026-03-03 00:30:29 | INFO     | __main__ |   MAPE: 16.795
2026-03-03 00:30:29 | INFO     | __main__ |   MASE: 0.358
2026-03-03 00:30:29 | INFO     | __main__ | Fold 2/5: Dec 2018 -> Jan-May 2019
2026-03-03 00:30:29 | INFO     | __main__ |   Early stopped at epoch 35/150 (patience=10)
2026-03-03 00:30:29 | INFO     | __main__ |   Best val loss: 0.005544
2026-03-03 00:30:29 | INFO     | __main__ |   RMSE: 50.309
2026-03-03 00:30:29 | INFO     | __main__ |   MAE: 47.261
2026-03-03 00:30:29 | INFO     | __main__ |   MAPE: 61.375
2026-03-03 00:30:29 | INF

In [17]:
# LSTM Diagnostic Plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Actual vs Predicted
ax = axes[0]
ax.plot(test_ts.index, test_ts.values, 'k-o', markersize=5, linewidth=2, label='Actual')
ax.plot(test_ts.index, lstm_pred, '#55A868', linestyle='--', marker='D', markersize=5,
        linewidth=1.5, label='LSTM')
ax.set_title(f'LSTM: Actual vs Predicted (RMSE={lstm_test_metrics["RMSE"]:.2f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

# 2. Residuals
ax = axes[1]
lstm_resid = test_ts.values - lstm_pred
months = [d.strftime('%b\n%Y') for d in test_ts.index]
ax.bar(months, lstm_resid, color='#55A868', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_title('LSTM Residuals')
ax.set_ylabel('Residual')
ax.tick_params(axis='x', rotation=45, labelsize=7)

# 3. Absolute Error by Month
ax = axes[2]
ax.bar(months, np.abs(lstm_resid), color='#55A868', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_title('LSTM Absolute Error by Month')
ax.set_ylabel('Absolute Error')
ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.suptitle('LSTM Diagnostic Plots', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_lstm_diagnostics.png', dpi=150, bbox_inches='tight')
plt.close('all')
import gc; gc.collect()

7373

---

## 8. Model Comparison

In [18]:
# Comparison table: CV metrics + Final fold metrics
comparison_data = {
    'Model': ['Holt-Winters', 'XGBoost', 'Random Forest', 'LSTM'],
    'RMSE (CV)': [f"{hw_cv_summary['RMSE']['mean']:.2f} +/- {hw_cv_summary['RMSE']['std']:.2f}",
                  f"{xgb_cv_summary['RMSE']['mean']:.2f} +/- {xgb_cv_summary['RMSE']['std']:.2f}",
                  f"{rf_cv_summary['RMSE']['mean']:.2f} +/- {rf_cv_summary['RMSE']['std']:.2f}",
                  f"{lstm_cv_summary['RMSE']['mean']:.2f} +/- {lstm_cv_summary['RMSE']['std']:.2f}"],
    'MAE (CV)': [f"{hw_cv_summary['MAE']['mean']:.2f} +/- {hw_cv_summary['MAE']['std']:.2f}",
                 f"{xgb_cv_summary['MAE']['mean']:.2f} +/- {xgb_cv_summary['MAE']['std']:.2f}",
                 f"{rf_cv_summary['MAE']['mean']:.2f} +/- {rf_cv_summary['MAE']['std']:.2f}",
                 f"{lstm_cv_summary['MAE']['mean']:.2f} +/- {lstm_cv_summary['MAE']['std']:.2f}"],
    'RMSE (Final)': [hw_test_metrics['RMSE'], xgb_test_metrics['RMSE'],
                     rf_test_metrics['RMSE'], lstm_test_metrics['RMSE']],
    'MAE (Final)': [hw_test_metrics['MAE'], xgb_test_metrics['MAE'],
                    rf_test_metrics['MAE'], lstm_test_metrics['MAE']],
    'MAPE (Final)': [hw_test_metrics['MAPE'], xgb_test_metrics['MAPE'],
                     rf_test_metrics['MAPE'], lstm_test_metrics['MAPE']],
    'MASE (Final)': [hw_test_metrics['MASE'], xgb_test_metrics['MASE'],
                     rf_test_metrics['MASE'], lstm_test_metrics['MASE']],
}
comparison_df = pd.DataFrame(comparison_data).set_index('Model')

print('Test Set Performance Comparison (5-Fold Walk-Forward CV + Final Fold)')
print('=' * 90)
print(comparison_df.to_string())
print(f'\nBest model by CV RMSE: {min(["Holt-Winters", "XGBoost", "Random Forest", "LSTM"], key=lambda m: [hw_cv_summary, xgb_cv_summary, rf_cv_summary, lstm_cv_summary][["Holt-Winters", "XGBoost", "Random Forest", "LSTM"].index(m)]["RMSE"]["mean"])}')

Test Set Performance Comparison (5-Fold Walk-Forward CV + Final Fold)
                     RMSE (CV)         MAE (CV)  RMSE (Final)  MAE (Final)  MAPE (Final)  MASE (Final)
Model                                                                                                 
Holt-Winters   23.04 +/- 11.74  20.80 +/- 11.81     37.717903    35.326256     62.219464      2.403342
XGBoost         17.90 +/- 8.95   15.30 +/- 7.73     21.449183    19.763045     35.512440      1.344534
Random Forest   18.06 +/- 9.77   15.43 +/- 8.38     25.724686    21.202611     43.139616      1.442472
LSTM           27.48 +/- 15.30  24.38 +/- 14.40     33.798484    29.782285     49.616287      2.026171

Best model by CV RMSE: XGBoost


In [19]:
# Overlay plot: All 4 models' predictions vs actual on test set
fig, ax = plt.subplots(figsize=(14, 7))

# Training data (last 2 years)
ax.plot(train_ts.index[-24:], train_ts.values[-24:], 'gray', linewidth=1.5,
        alpha=0.5, label='Training Data')

# Actual test
ax.plot(test_ts.index, test_ts.values, 'k-o', markersize=6, linewidth=2, label='Actual')

# Model predictions
ax.plot(test_ts.index, hw_pred.values, '#4C72B0', linestyle='--', linewidth=1.5,
        marker='o', markersize=4, label=f'Holt-Winters (RMSE={hw_test_metrics["RMSE"]:.2f})')
ax.plot(test_ts.index, xgb_pred, '#DD8452', linestyle='-.',  linewidth=1.5,
        marker='s', markersize=4, label=f'XGBoost (RMSE={xgb_test_metrics["RMSE"]:.2f})')
ax.plot(test_ts.index, rf_pred, '#8B4513', linestyle=':',  linewidth=2,
        marker='D', markersize=4, label=f'RF (RMSE={rf_test_metrics["RMSE"]:.2f})')
ax.plot(test_ts.index, lstm_pred, '#55A868', linestyle='-', linewidth=1.5,
        marker='^', markersize=4, label=f'LSTM (RMSE={lstm_test_metrics["RMSE"]:.2f})')

ax.axvline(x=test_ts.index[0], color='gray', linestyle='--', alpha=0.3)
ax.set_title('All Models: Predictions vs Actual (Test Set)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Water Inflow')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

0

In [20]:
# Residual comparison (4 models)
# Compute residuals (final fold)
hw_resid_test = test_ts.values - hw_pred.values
xgb_resid_test = test_ts.values - xgb_pred
rf_resid_test = test_ts.values - rf_pred
lstm_resid_test = test_ts.values - lstm_pred

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

residuals_dict = {
    'Holt-Winters': hw_resid_test,
    'XGBoost': xgb_resid_test,
    'RF': rf_resid_test,
    'LSTM': lstm_resid_test
}


colors = ['#4C72B0', '#DD8452', '#8B4513', '#55A868']

for ax, (name, resid), color in zip(axes, residuals_dict.items(), colors):
    months = [d.strftime('%b\n%Y') for d in test_ts.index]
    ax.bar(months, resid, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{name} Residuals')
    ax.set_ylabel('Residual (Actual - Predicted)')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    
    ax.text(0.02, 0.98, f'Mean: {np.mean(resid):.2f}\nStd: {np.std(resid):.2f}',
            transform=ax.transAxes, fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Test Set Residual Comparison (4 Models)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_residual_comparison.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

13499

In [21]:
# Error vs Forecast Horizon Analysis (4 models)
horizons = np.arange(1, TEST_SIZE + 1)

hw_abs_errors = np.abs(hw_resid_test)
xgb_abs_errors = np.abs(xgb_resid_test)
rf_abs_errors = np.abs(rf_resid_test)
lstm_abs_errors = np.abs(lstm_resid_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Absolute error by horizon
ax = axes[0]
ax.plot(horizons, hw_abs_errors, color='#4C72B0', linewidth=2,
        marker='o', markersize=5, label='Holt-Winters')
ax.plot(horizons, xgb_abs_errors, color='#DD8452', linewidth=2,
        marker='s', markersize=5, label='XGBoost')
ax.plot(horizons, rf_abs_errors, color='#8B4513', linewidth=2,
        marker='D', markersize=5, label='RF')
ax.plot(horizons, lstm_abs_errors, color='#55A868', linewidth=2,
        marker='^', markersize=5, label='LSTM')
ax.set_title('Absolute Error vs Forecast Horizon')
ax.set_xlabel('Forecast Horizon (months ahead)')
ax.set_ylabel('Absolute Error')
ax.set_xticks(horizons)
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Cumulative RMSE by horizon
ax = axes[1]
hw_cum_rmse = [np.sqrt(np.mean(hw_resid_test[:h]**2)) for h in range(1, TEST_SIZE + 1)]
xgb_cum_rmse = [np.sqrt(np.mean(xgb_resid_test[:h]**2)) for h in range(1, TEST_SIZE + 1)]
rf_cum_rmse = [np.sqrt(np.mean(rf_resid_test[:h]**2)) for h in range(1, TEST_SIZE + 1)]
lstm_cum_rmse = [np.sqrt(np.mean(lstm_resid_test[:h]**2)) for h in range(1, TEST_SIZE + 1)]

ax.plot(horizons, hw_cum_rmse, color='#4C72B0', linewidth=2,
        marker='o', markersize=5, label='Holt-Winters')
ax.plot(horizons, xgb_cum_rmse, color='#DD8452', linewidth=2,
        marker='s', markersize=5, label='XGBoost')
ax.plot(horizons, rf_cum_rmse, color='#8B4513', linewidth=2,
        marker='D', markersize=5, label='RF')
ax.plot(horizons, lstm_cum_rmse, color='#55A868', linewidth=2,
        marker='^', markersize=5, label='LSTM')
ax.set_title('Cumulative RMSE vs Forecast Horizon')
ax.set_xlabel('Forecast Horizon (months ahead)')
ax.set_ylabel('Cumulative RMSE')
ax.set_xticks(horizons)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_error_vs_horizon.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

# Print short-horizon summary
print('\nShort-Horizon Analysis (1-5 months ahead):')
print('=' * 70)
for h in [1, 2, 3, 4, 5]:
    hw = np.sqrt(np.mean(hw_resid_test[:h]**2))
    x = np.sqrt(np.mean(xgb_resid_test[:h]**2))
    r = np.sqrt(np.mean(rf_resid_test[:h]**2))
    l = np.sqrt(np.mean(lstm_resid_test[:h]**2))
    best = min([(hw,'HW'),(x,'XGBoost'),(r,'RF'),(l,'LSTM')])
    print(f'  h={h}: HW={hw:.3f} | XGBoost={x:.3f} | RF={r:.3f} | LSTM={l:.3f} | Best: {best[1]}')


Short-Horizon Analysis (1-5 months ahead):
  h=1: HW=22.395 | XGBoost=15.055 | RF=8.414 | LSTM=13.272 | Best: RF
  h=2: HW=31.854 | XGBoost=19.554 | RF=17.117 | LSTM=23.398 | Best: RF
  h=3: HW=42.064 | XGBoost=25.587 | RF=21.700 | LSTM=36.894 | Best: RF
  h=4: HW=40.855 | XGBoost=23.086 | RF=18.840 | LSTM=37.293 | Best: RF
  h=5: HW=37.718 | XGBoost=21.449 | RF=25.725 | LSTM=33.798 | Best: XGBoost


In [22]:
# Save summary figure with all key plots (4 models)
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

# Row 1: Individual model predictions
models_info = [
    ('Holt-Winters', hw_pred.values, hw_test_metrics, '#4C72B0'),
    ('XGBoost', xgb_pred, xgb_test_metrics, '#DD8452'),
    ('RF', rf_pred, rf_test_metrics, '#8B4513'),
    ('LSTM', lstm_pred, lstm_test_metrics, '#55A868')
]

for ax, (name, pred, met, color) in zip(axes[0], models_info):
    ax.plot(test_ts.index, test_ts.values, 'k-o', markersize=3, linewidth=1.5, label='Actual')
    ax.plot(test_ts.index, pred, color=color, linestyle='--', marker='s', markersize=3,
            linewidth=1.5, label=f'{name}')
    ax.set_title(f'{name}\nRMSE={met["RMSE"]:.2f} | MAE={met["MAE"]:.2f} | MAPE={met["MAPE"]:.1f}%',
                fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Row 2: Comparison plots
# Overlay
ax = axes[1, 0]
ax.plot(test_ts.index, test_ts.values, 'k-o', markersize=3, linewidth=2, label='Actual')
ax.plot(test_ts.index, hw_pred.values, '#4C72B0', linestyle='--', linewidth=1.2, label='HW')
ax.plot(test_ts.index, xgb_pred, '#DD8452', linestyle='-.', linewidth=1.2, label='XGBoost')
ax.plot(test_ts.index, rf_pred, '#8B4513', linestyle=':', linewidth=1.5, label='RF')
ax.plot(test_ts.index, lstm_pred, '#55A868', linestyle='-', linewidth=1.2, label='LSTM')
ax.set_title('All Models Overlay', fontsize=9)
ax.legend(fontsize=6)
ax.tick_params(axis='x', rotation=45, labelsize=6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Error by horizon
ax = axes[1, 1]
ax.plot(horizons, hw_abs_errors, '#4C72B0', marker='o', markersize=3, label='HW')
ax.plot(horizons, xgb_abs_errors, '#DD8452', marker='s', markersize=3, label='XGBoost')
ax.plot(horizons, rf_abs_errors, '#8B4513', marker='D', markersize=3, label='RF')
ax.plot(horizons, lstm_abs_errors, '#55A868', marker='^', markersize=3, label='LSTM')
ax.set_title('Abs Error by Horizon', fontsize=9)
ax.set_xlabel('Months Ahead')
ax.legend(fontsize=6)

# Metrics bar chart
ax = axes[1, 2]
x = np.arange(4)
width = 0.2
metrics_vals = [
    [hw_test_metrics[m] for m in ['RMSE', 'MAE', 'MAPE', 'MASE']],
    [xgb_test_metrics[m] for m in ['RMSE', 'MAE', 'MAPE', 'MASE']],
    [rf_test_metrics[m] for m in ['RMSE', 'MAE', 'MAPE', 'MASE']],
    [lstm_test_metrics[m] for m in ['RMSE', 'MAE', 'MAPE', 'MASE']]
]
ax.bar(x - 1.5*width, metrics_vals[0], width, color='#4C72B0', alpha=0.8, label='HW')
ax.bar(x - 0.5*width, metrics_vals[1], width, color='#DD8452', alpha=0.8, label='XGBoost')
ax.bar(x + 0.5*width, metrics_vals[2], width, color='#8B4513', alpha=0.8, label='RF')
ax.bar(x + 1.5*width, metrics_vals[3], width, color='#55A868', alpha=0.8, label='LSTM')
ax.set_xticks(x)
ax.set_xticklabels(['RMSE', 'MAE', 'MAPE (%)', 'MASE'], fontsize=8)
ax.set_title('Metrics Comparison', fontsize=9)
ax.legend(fontsize=6)

# Cumulative RMSE
ax = axes[1, 3]
ax.plot(horizons, hw_cum_rmse, '#4C72B0', marker='o', markersize=3, label='HW')
ax.plot(horizons, xgb_cum_rmse, '#DD8452', marker='s', markersize=3, label='XGBoost')
ax.plot(horizons, rf_cum_rmse, '#8B4513', marker='D', markersize=3, label='RF')
ax.plot(horizons, lstm_cum_rmse, '#55A868', marker='^', markersize=3, label='LSTM')
ax.set_title('Cumulative RMSE', fontsize=9)
ax.set_xlabel('Months Ahead')
ax.legend(fontsize=6)

plt.suptitle('Water Inflow Forecasting - 4 Model Comparison Summary',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'c2_model_comparison_summary.png', dpi=150, bbox_inches='tight')
plt.close("all")
import gc; gc.collect()

6743

### Walk-Forward CV: All 5-Year Predictions

Combined visualization of all 5 walk-forward CV folds (2018-2022) showing each model's predictions against actual values.

In [23]:
# Walk-Forward CV: All 5-Year Predictions (2018-2022)
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Walk-Forward Cross-Validation: 5-Year Predictions (2018-2022)',
             fontsize=16, fontweight='bold', y=1.02)

model_colors = {
    'Holt-Winters': '#2196F3',
    'XGBoost': '#FF9800',
    'Random Forest': '#795548',
    'LSTM': '#4CAF50'
}

fold_rmses = {name: [] for name in model_colors}

for fold_idx in range(len(cv_folds)):
    row, col = divmod(fold_idx, 3)
    ax = axes[row, col]
    _, test_fold, label = cv_folds[fold_idx]
    test_year = test_fold.index[0].year

    # Actual values
    months = [d.strftime('%b') for d in test_fold.index]
    ax.plot(months, test_fold.values, 'ko-', linewidth=2, markersize=6, label='Actual', zorder=5)

    # Holt-Winters
    hw_p = hw_fold_predictions[fold_idx]['pred']
    ax.plot(months, hw_p.values, 's--', color=model_colors['Holt-Winters'],
            linewidth=1.5, markersize=5, label='HW')
    fold_rmses['Holt-Winters'].append(np.sqrt(np.mean((test_fold.values - hw_p.values)**2)))

    # XGBoost
    xgb_p = xgb_fold_predictions[fold_idx]['pred']
    ax.plot(months, xgb_p, 'D--', color=model_colors['XGBoost'],
            linewidth=1.5, markersize=5, label='XGB')
    fold_rmses['XGBoost'].append(np.sqrt(np.mean((test_fold.values - np.array(xgb_p))**2)))

    # Random Forest
    rf_p = rf_fold_predictions[fold_idx]['pred']
    ax.plot(months, rf_p, '^--', color=model_colors['Random Forest'],
            linewidth=1.5, markersize=5, label='RF')
    fold_rmses['Random Forest'].append(np.sqrt(np.mean((test_fold.values - np.array(rf_p))**2)))

    # LSTM
    lstm_p = lstm_fold_predictions[fold_idx]['pred']
    ax.plot(months, lstm_p, 'v--', color=model_colors['LSTM'],
            linewidth=1.5, markersize=5, label='LSTM')
    fold_rmses['LSTM'].append(np.sqrt(np.mean((test_fold.values - np.array(lstm_p))**2)))

    ax.set_title(f'{test_year} (Fold {fold_idx+1})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Inflow')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

# Subplot 6: Per-fold RMSE bar chart
ax_bar = axes[1, 2]
x = np.arange(len(cv_folds))
bar_width = 0.2
for i, (name, color) in enumerate(model_colors.items()):
    ax_bar.bar(x + i * bar_width, fold_rmses[name], bar_width,
              label=name, color=color, alpha=0.85)
ax_bar.set_xticks(x + 1.5 * bar_width)
ax_bar.set_xticklabels([f'{2018+i}' for i in range(len(cv_folds))])
ax_bar.set_xlabel('Test Year')
ax_bar.set_ylabel('RMSE')
ax_bar.set_title('Per-Fold RMSE by Model', fontsize=12, fontweight='bold')
ax_bar.legend(fontsize=8)
ax_bar.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig('../figures/c2_walkforward_all_folds.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close(fig)
print('Saved: figures/c2_walkforward_all_folds.png')


Saved: figures/c2_walkforward_all_folds.png


---

## 9. Summary

| Aspect | Holt-Winters | XGBoost | Random Forest | LSTM |
|--------|-------------|---------|---------------|------|
| **Type** | Statistical | ML (Boosting) | ML (Bagging) | Deep Learning |
| **Interpretability** | High | Medium | Medium | Low |
| **Feature Engineering** | None | Extensive | Extensive (shared) | None (raw sequence) |
| **Overfitting Risk** | Low | Medium | Low | Medium |

### Validation Strategy

- **Walk-Forward CV**: 5-fold expanding window, anchored on December
- **Forecast Horizon**: 5 months (January-May), as required by the assignment
- **Metrics**: Per-fold RMSE, MAE, MAPE, MASE + mean/std across folds

In [24]:
# Final summary printout
print('=' * 70)
print('CASE 2: WATER INFLOW FORECASTING - MODEL DEVELOPMENT COMPLETE')
print('=' * 70)
print(f'\nDataset: {len(ts)} monthly observations ({ts.index[0].strftime("%Y-%m")} to {ts.index[-1].strftime("%Y-%m")})')
print(f'Validation: 5-fold walk-forward CV (December-anchored, 5-month horizon)')
print(f'Final test: {test_ts.index[0].strftime("%Y-%m")} to {test_ts.index[-1].strftime("%Y-%m")} ({TEST_SIZE} months)')

print(f'\nCV Performance Summary (mean +/- std across 5 folds):')
print('-' * 60)
for name, cv_summary in [('Holt-Winters', hw_cv_summary), ('XGBoost', xgb_cv_summary),
                          ('Random Forest', rf_cv_summary), ('LSTM', lstm_cv_summary)]:
    rmse = cv_summary['RMSE']
    print(f'  {name:>15s}: RMSE = {rmse["mean"]:.3f} +/- {rmse["std"]:.3f}')

print(f'\nFinal Test Set Results (Fold 5):')
print(comparison_df[['RMSE (Final)', 'MAE (Final)', 'MAPE (Final)']].to_string())
print(f'\nAll figures saved to: {os.path.abspath(FIGURES_DIR)}')

saved_figures = [f for f in os.listdir(FIGURES_DIR) if f.startswith('c2_') and f.endswith('.png')]
print(f'\nSaved figures ({len(saved_figures)}):')
for fig_name in sorted(saved_figures):
    print(f'  - {fig_name}')

CASE 2: WATER INFLOW FORECASTING - MODEL DEVELOPMENT COMPLETE

Dataset: 281 monthly observations (1999-01 to 2022-05)
Validation: 5-fold walk-forward CV (December-anchored, 5-month horizon)
Final test: 2022-01 to 2022-05 (5 months)

CV Performance Summary (mean +/- std across 5 folds):
------------------------------------------------------------
     Holt-Winters: RMSE = 23.036 +/- 11.743
          XGBoost: RMSE = 17.896 +/- 8.946
    Random Forest: RMSE = 18.062 +/- 9.767
             LSTM: RMSE = 27.478 +/- 15.305

Final Test Set Results (Fold 5):
               RMSE (Final)  MAE (Final)  MAPE (Final)
Model                                                 
Holt-Winters      37.717903    35.326256     62.219464
XGBoost           21.449183    19.763045     35.512440
Random Forest     25.724686    21.202611     43.139616
LSTM              33.798484    29.782285     49.616287

All figures saved to: /Users/srk/VsProjects/water-inflow-forecasting/figures

Saved figures (27):
  - c2_acf_pacf